# Disk-Resident Dataset Overview

This notebook expands the dataset-side view of the disk-resident release package. It mirrors the role of `memory_resident_ann/data/dataset_overview.ipynb`, but focuses on the dataset registry, the QSO static layout boundary, the RDO dynamic layout boundary, and the baseline-specific artifact contracts used by the five disk-resident systems.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

DATA_DIR = Path.cwd()
CONFIG = DATA_DIR / "config.yml"
with CONFIG.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset_manifest = cfg["dataset"]["manifest"]
dataset_groups = cfg["dataset"]["groups"]
baselines = cfg["baselines"]


## Dataset Registry

The table below collects the datasets prepared by the disk-resident package. The first group corresponds to the datasets already used in the QVLOF paper, while the second group adds the four 100M-scale datasets used by the disk-resident evaluation.

In [ ]:
rows = []
for name in cfg["dataset"]["prepare"]:
    meta = dataset_manifest[name]
    rows.append({
        "dataset": name,
        "paper_label": meta.get("paper_label", name),
        "group": meta.get("group", ""),
        "scale": meta.get("scale", ""),
        "value_type": meta["value_type"],
        "distance": meta["distance"],
        "base_file": meta["base_file"],
        "query_file": meta["query_file"],
        "groundtruth_file": meta["groundtruth_file"],
        "note": meta.get("note", ""),
    })
pd.DataFrame(rows)

## Dataset Group Structure

The release package separates paper datasets from the larger 100M extension group so that the same registry can support both the original QVLOF evaluation and the disk-resident extension figures.

In [ ]:
rows = []
for name, meta in dataset_groups.items():
    rows.append({
        "group": name,
        "description": meta["description"],
        "dataset_count": len(meta["datasets"]),
        "datasets": ", ".join(meta["datasets"]),
    })
pd.DataFrame(rows)

## Access-Profile Contract

QSO does not consume a runtime ANN index. It consumes the base vectors together with a workload-side access profile. The access profile remains dataset-aligned and vector-id aligned, so the resulting QSO permutation can later be translated into the physical layout unit expected by each disk-resident baseline.

In [ ]:
access_rows = []
for column in cfg["access_profile"]["columns"]:
    access_rows.append({
        "field": column,
        "semantics": cfg["access_profile"]["semantics"].get(column, "required access-profile field"),
    })
pd.DataFrame(access_rows)

## Workload-Window Contract

RDO extends the static QSO view into a sequence of workload windows. Each window carries a candidate layout family and the metadata needed by the switch policy to pick one active disk layout at a time.

In [ ]:
window_rows = [{"field": field} for field in cfg["workload_windows"]["fields"]]
pd.DataFrame(window_rows)

## Static QSO Artifact Layer

The next table shows how the dataset-side QSO output is translated into a baseline-specific layout artifact. The search path itself is not replaced. Instead, QSO controls the order or grouping of the physical unit that each baseline already knows how to build and search.

In [ ]:
rows = []
for name, meta in baselines.items():
    static_meta = meta["static_qso"]
    rows.append({
        "baseline": name,
        "layout_unit": meta["layout_unit"],
        "data_fields": ", ".join(meta["data_fields"]),
        "artifact_names": ", ".join(static_meta["artifact_names"]),
        "import_stage": static_meta["import_stage"],
        "rebuild_output": ", ".join(static_meta["rebuild_output"]),
    })
pd.DataFrame(rows)

## Dynamic RDO Artifact Layer

RDO introduces a time-varying layout family. The disk-resident package records which files or directories become the switchable unit for each baseline, and which replay target is evaluated when the workload moves from one window to another.

In [ ]:
rows = []
for name, meta in baselines.items():
    dynamic_meta = meta["dynamic_rdo"]
    rows.append({
        "baseline": name,
        "layout_family": ", ".join(dynamic_meta["layout_family"]),
        "switch_unit": dynamic_meta["switch_unit"],
        "replay_target": dynamic_meta["replay_target"],
    })
pd.DataFrame(rows)

## Baseline Data Wiring Matrix

This matrix emphasizes that each baseline consumes the same dataset registry through a different index-specific field set. The dataset-side layer is shared, while the import hook and physical rebuild outputs remain system-specific.

In [ ]:
rows = []
for name, meta in baselines.items():
    rows.append({
        "baseline": name,
        "upstream": meta["upstream"],
        "layout_unit": meta["layout_unit"],
        "dataset_fields": len(meta["data_fields"]),
        "static_artifacts": len(meta["static_qso"]["artifact_names"]),
        "dynamic_layout_family_entries": len(meta["dynamic_rdo"]["layout_family"]),
    })
pd.DataFrame(rows)

## Per-Baseline Static and Dynamic Coupling

The following expanded rows are useful when reading the project-level integration notes under `../index/*/README.md`. They summarize, in one place, what the dataset-side artifact contract looks like before the baseline-specific build path takes over.

In [ ]:
rows = []
for name, meta in baselines.items():
    rows.append({
        "baseline": name,
        "static_import_stage": meta["static_qso"]["import_stage"],
        "static_artifact_names": ", ".join(meta["static_qso"]["artifact_names"]),
        "dynamic_switch_unit": meta["dynamic_rdo"]["switch_unit"],
        "dynamic_replay_target": meta["dynamic_rdo"]["replay_target"],
    })
pd.DataFrame(rows)

## Requirements Summary

The dataset registry also carries the minimal environment grouping needed to consume the prepared artifacts. This keeps the data-side notebook connected to the dependency structure summarized under `../requirements/README.md`.

In [ ]:
rows = []
for group_name, items in cfg["requirements"]["system_groups"].items():
    rows.append({
        "system": group_name,
        "requirement_count": len(items),
        "requirements": ", ".join(items),
    })
pd.DataFrame(rows)

## Notes

This notebook stays at the dataset and artifact-registration layer. Code-level coupling details remain in the baseline-specific integration notes, but every row here is already aligned with those project-specific import stages and rebuild outputs.